In [1]:
# Load packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics.pairwise import euclidean_distances

from FDApy import DenseFunctionalData, MultivariateFunctionalData
from FDApy.representation import DenseArgvals, DenseValues
from FDApy.preprocessing import MFPCA
from FDApy.visualization import plot, plot_multivariate

from nba import NbaScraper, ShotCharts

plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r'\usepackage{amsfonts}'
})

Matplotlib is building the font cache; this may take a moment.
Cannot decode configuration file PosixPath('/Volumes/Samsung_T5/research/10_in_progress/shooting_nba_fda/code/.venv/lib/python3.11/site-packages/matplotlib/mpl-data/stylelib/.__classic_test_patch.mplstyle') as utf-8.


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 37: invalid start byte

In [2]:
# Load data
with open('./data/players_shots_density_attempted.pickle', 'rb') as f:
    players_shots_density = pickle.load(f)
with open('./data/players_shots_density_made.pickle', 'rb') as f:
    players_shots_density_made = pickle.load(f)
with open('./data/player_position.pickle', 'rb') as f:
    players_position = pickle.load(f)

# Load MFPCA results
with open('./data/MFPCA.pickle', 'rb') as f:
    mfpca = pickle.load(f)
with open('./data/scores.pickle', 'rb') as f:
    scores = pickle.load(f)
with open('./data/MFPCA_reconstruction.pickle', 'rb') as f:
    fdata_reconstruction = pickle.load(f)

In [3]:
# Reshape scores
scores = pd.DataFrame(scores)
scores.insert(loc=0, column='PLAYER_ID', value=players_shots_density.PLAYER_ID.values)
scores.insert(loc=0, column='PLAYER_NAME', value=players_shots_density.PLAYER_NAME.values)
scores = scores.join(players_position.set_index('PLAYER_ID'), how='left', on='PLAYER_ID', rsuffix='_')

In [4]:
scores_mat = scores[[0, 1, 2, 3]].values

In [5]:
scaler = StandardScaler()
scaler.fit(scores_mat)
new_scores = scaler.transform(scores_mat)

In [12]:
# Compute distances betwween scores
distance_scores = euclidean_distances(new_scores)

In [6]:
# K-means
kmeans = KMeans(n_clusters=5).fit(new_scores)

In [7]:
results = {
    'centers': kmeans.cluster_centers_,
    'pred': kmeans.predict(new_scores)
}

In [13]:
with open('./data/distance_scores.pickle', 'wb') as f:
    pickle.dump(distance_scores, f, protocol=pickle.HIGHEST_PROTOCOL)

In [8]:
with open('./data/clustering.pickle', 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

In [14]:
# Weighted K-means
pct = mfpca.eigenvalues / np.sum(mfpca.eigenvalues)
weighted_scores = np.multiply(new_scores, pct[:4])

In [16]:
# Compute distances betwween weighted scores
distance_weighted_scores = euclidean_distances(weighted_scores)

In [10]:
kmeans = KMeans(n_clusters=5).fit(weighted_scores)

In [11]:
results = {
    'centers': kmeans.cluster_centers_,
    'pred': kmeans.predict(weighted_scores)
}

In [17]:
with open('./data/distance_weighted_scores.pickle', 'wb') as f:
    pickle.dump(distance_weighted_scores, f, protocol=pickle.HIGHEST_PROTOCOL)

In [12]:
with open('./data/clustering_weighted.pickle', 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

In [13]:
# K-medoids
kmedoids = KMedoids(n_clusters=5).fit(new_scores)

In [14]:
results = {
    'centers': kmedoids.cluster_centers_,
    'idx_centers': [np.argmax(np.isclose(new_scores[:, 0], kmedoids.cluster_centers_[idx, 0])) for idx in range(5)],
    'pred': kmedoids.predict(new_scores)
}

In [15]:
with open('./data/clustering_kmedoids.pickle', 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

In [16]:
# Weighted K-medoids
kmedoids = KMedoids(n_clusters=5).fit(weighted_scores)

In [17]:
results = {
    'centers': kmedoids.cluster_centers_,
    'idx_centers': [np.argmax(np.isclose(weighted_scores[:, 0], kmedoids.cluster_centers_[idx, 0])) for idx in range(5)],
    'pred': kmedoids.predict(weighted_scores)
}

In [18]:
with open('./data/clustering_kmedoids_weighted.pickle', 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)